# Chain Domain Finder — 3 Domain Pipeline
## Algorithm
### Step 1
Query domain **A** → DOMIN encoder → FAISS → **top-n candidates** {B0, B1, ..., Bn-1}

### Step 2 — Compatibility Metric
- For each **Bi**, retrieve top-10 from FAISS → average score = **threshold_i**
- Compute **score(Bi, Bj)** = DOMIN inner_product(query_emb(Bi), key_emb(Bj))
- If **score(Bi, Bj) > threshold_i** → Bi and Bj are compatible

### Step 3
Filter compatible pairs → output (A, Bi, Bj) triples

---

## Setup — Import Modules

In [ ]:
import os, sys, torch, faiss, numpy as np, logging

sys.path.insert(0, "/sujin/PycharmProjects/DOMINO")
sys.path.insert(0, "/sujin/PycharmProjects/DOMINO/src/DOMO/models")
sys.path.insert(0, "/sujin/PycharmProjects/DOMINO/src/DOMO")
sys.path.insert(0, "/sujin/PycharmProjects/DOMINO/src/DOMIN")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

## Utility Functions

In [ ]:
def sa_to_aa(sa: str) -> str:
    """One SA = one domain. Split by <unk>, extract AA from each segment, rejoin with <unk>."""
    fragments = [p[::2] for p in sa.split("<unk>") if p]
    return "<unk>".join(fragments)

## Load DOMIN + FAISS

In [ ]:
DEVICE = "cuda"

FAISS_IDX = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/key_embeddings.index"
FAISS_TSV = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/sampled_proteins.tsv"

from src.DOMIN.models.ted.ted_domain_model import TedDomainModel

log.info("Loading DOMIN...")
domin = TedDomainModel(
    config_path="/sujin/PycharmProjects/DOMINO/weights/DOMIN",
    from_checkpoint="/sujin/PycharmProjects/DOMINO/weights/DOMIN/DOMIN.pt"
)
domin.to(DEVICE).eval()
TEMP = domin.model.temperature.item()
log.info(f"  DOMIN loaded (temp={TEMP:.4f})")

log.info("Loading FAISS index...")
faiss_index = faiss.read_index(FAISS_IDX, faiss.IO_FLAG_MMAP)
faiss_index.nprobe = 10000
idx_to_segment = [l.strip().split("\t")[1] for l in open(FAISS_TSV)]
log.info(f"  FAISS: {faiss_index.ntotal:,} vectors | Segments: {len(idx_to_segment):,}")

## Step 1 — Query A → top-n Candidates

Set `QUERY_SA` and `TOP_N` here, then run.

In [ ]:
QUERY_SA = "GvAlEvKvLlLvEvRlKvNvEvMlDvRvLlAvWvQvIlGvRvYcRpKpNvGhEdPnKcLvVvLcKvDvQpPvDvMvLsAvGvYnEvAvLsLvAvQvFnKvKvLsEvQvEvTsSvRvLsTvVvAvRvNvTvRcNvIvNv"
TOP_N = 10

log.info(f"[Step 1] Query SA: {QUERY_SA[:60]}...")

with torch.no_grad():
    q = domin.get_query_repr(QUERY_SA).cpu().numpy()

distances, indices = faiss_index.search(
    q.reshape(1, -1).astype("float32"), TOP_N * 3
)
scores = distances[0] / TEMP

# Collect unique candidates (dedupe by AA content)
candidates = []
seen_aa = set()
for idx, score in zip(indices[0].tolist(), scores.tolist()):
    if idx == -1 or score == -1:
        continue
    seg = idx_to_segment[idx] if idx < len(idx_to_segment) else None
    if not seg:
        continue
    # Use AA string (sa_to_aa) as dedup key
    aa_str = sa_to_aa(seg)
    if aa_str in seen_aa:
        continue
    seen_aa.add(aa_str)
    candidates.append((int(idx), float(score), str(seg), aa_str))
    if len(candidates) >= TOP_N:
        break

candidates.sort(key=lambda x: x[1], reverse=True)
log.info(f"  Retrieved {len(candidates)} unique candidates")

print("\n===== Candidates =====")
for k, (idx, score, seg, aa_str) in enumerate(candidates):
    print(f"[{k}] idx={idx:,}  score={score:.4f}  AA={aa_str[:30]}...")

## Step 2 — Pairwise Compatibility

For each candidate Bi:
1. Encode Bi → query_emb(Bi)
2. FAISS top-10 retrieval → average score = **threshold_i**
3. Compute inner_product(query_emb(Bi), key_emb(Bj)) for all Bj ≠ Bi
4. compatible if inner_product > threshold_i

In [ ]:
TOP_K = 10

log.info(f"[Step 2] Pairwise compatibility among {len(candidates)} candidates...")

# Compute threshold_i for each candidate: avg FAISS top-K score
thresholds = {}
for i, (idx_i, score_i, seg_i, aa_i) in enumerate(candidates):
    with torch.no_grad():
        q_emb = domin.get_query_repr(seg_i).cpu().numpy()
    distances_k, _ = faiss_index.search(
        q_emb.reshape(1, -1).astype("float32"), TOP_K
    )
    thresholds[i] = float(np.mean(distances_k[0] / TEMP))
    log.info(f"  Candidate {i}: threshold_i = {thresholds[i]:.4f}")

# Compute pairwise inner products: query_emb(Bi) · key_emb(Bj)
pair_results = {}
for i in range(len(candidates)):
    for j in range(i + 1, len(candidates)):
        _, _, seg_i, aa_i = candidates[i]
        _, _, seg_j, aa_j = candidates[j]
        
        with torch.no_grad():
            query_emb_i = domin.get_query_repr(seg_i).cpu().numpy()
            key_emb_j   = domin.get_key_repr(seg_j).cpu().numpy()
        
        score_ij = float(np.dot(query_emb_i, key_emb_j))
        
        pair_results[(i, j)] = {
            'score_ij': score_ij,
            'threshold_i': thresholds[i],
            'compatible': score_ij > thresholds[i],
            'aa_i': aa_i,
            'aa_j': aa_j,
        }

n_compat = sum(1 for v in pair_results.values() if v['compatible'])
log.info(f"  {n_compat}/{len(pair_results)} pairs are compatible")

print("\n===== Pairwise Results =====")
for (i, j), res in sorted(pair_results.items()):
    status = "✓ COMPATIBLE" if res['compatible'] else "✗ not compatible"
    print(f"({i}, {j}): score={res['score_ij']:.4f}  threshold={res['threshold_i']:.4f}  {status}")

## Step 3 — Filter Compatible Triples (A, Bi, Bj)

In [ ]:
TOP_M = 5

query_aa = sa_to_aa(QUERY_SA)

triples = []
for (i, j), res in pair_results.items():
    if not res['compatible']:
        continue
    idx_i, score_i, _, _ = candidates[i]
    idx_j, score_j, _, _ = candidates[j]
    triples.append({
        'rank': len(triples) + 1,
        'idx_i': idx_i,
        'idx_j': idx_j,
        'score_ij': res['score_ij'],
        'threshold_i': res['threshold_i'],
        'aa_A': query_aa,
        'aa_i': res['aa_i'],
        'aa_j': res['aa_j'],
    })

triples.sort(key=lambda x: x['score_ij'], reverse=True)
triples = triples[:TOP_M]

log.info(f"[Step 3] Found {len(triples)} compatible triples")

print("\n" + "=" * 70)
print("FINAL RESULTS — Compatible Triples (A, Bi, Bj)")
print("=" * 70)
if not triples:
    print("  No compatible triples found. Try increasing TOP_N or lowering TOP_K.")
else:
    for t in triples:
        print(f"\n--- Triple {t['rank']} ---")
        print(f"  A  : {t['aa_A']}")
        print(f"  Bi : idx={t['idx_i']:,}  score={t['score_ij']:.4f}  (threshold={t['threshold_i']:.4f})")
        print(f"  Bj : idx={t['idx_j']:,}")
        print(f"  Bi AA: {t['aa_i']}")
        print(f"  Bj AA: {t['aa_j']}")